# How to run sequential methods

In the previous tutorials, we have inferred the posterior using **amortized inference**. In **amortized inference**, we draw parameters from the prior, simulate the corresponding data, and then train a neural network to obtain the posterior. However, if one is interested in only one particular observation `x_o` sampling from the prior can be inefficient in the number of simulations because one is effectively learning a posterior estimate for all observations in the prior space. In this tutorial, we show how one can alleviate this issue by using **sequential methods** with `sbi`.

**Sequential methods** also starts by drawing parameters from the prior, simulating them, and training a neural network to estimate the posterior distribution. Afterwards, however, it continues inference in multiple rounds, focusing on a particular observation `x_o`. In each new round of inference, it draws samples from the obtained posterior distribution conditioned at `x_o` (instead of from the prior), simulates these, and trains the network again. This process can be repeated arbitrarily often to get increasingly good approximations to the true posterior distribution at `x_o`.

Running multi-round inference can be more efficient in the number of simulations, but it will lead to the posterior no longer being amortized (i.e. it will be accurate only for a specific observation `x_o`, not for any `x`).


## Main syntax


```python
inference = NPE(prior)
proposal = prior

for _ in range(num_rounds):
    theta = proposal.sample((100,))
    x = simulate(theta)

    # In `SNLE` and `SNRE`, you should not pass the `proposal` to `.append_simulations()`.
    density_estimator = inference.append_simulations(
        theta, x, proposal=proposal
    ).train()
    posterior = inference.build_posterior(density_estimator)
    proposal = posterior.set_default_x(x_o)
```

## Monitoring convergence across rounds

To decide how many rounds to run, you can measure how much each round changes the posterior estimate. `kl_divergence_mc` estimates the KL divergence between two posteriors from samples. In NPE-C, the proposal of a round is the posterior of the previous round, so you can compare the two directly in the loop:

```python
from sbi.diagnostics import kl_divergence_mc

inference = NPE(prior)
proposal = prior

for r in range(num_rounds):
    theta = proposal.sample((100,))
    x = simulate(theta)
    inference.append_simulations(theta, x, proposal=proposal).train()
    posterior = inference.build_posterior().set_default_x(x_o)
    if r > 0:
        kl, sem = kl_divergence_mc(posterior, proposal)
        print(f"Round {r}: KL = {kl:.3f} ± {sem:.3f}")
    proposal = posterior
```

In TSNPE, the proposal is a `RestrictedPrior`. Keep the previous posterior in a separate variable and compare against it.

Usually, the divergence decreases in the first rounds and then stays at a constant level. This level is not zero, because each round retrains the network. When the divergence no longer decreases, more rounds with the same number of simulations are unlikely to improve the estimate.

The divergence shows only whether consecutive rounds agree, not whether they are correct: the rounds can agree on an incorrect posterior. To check the final posterior at `x_o`, use [posterior predictive checks](https://sbi.readthedocs.io/en/latest/advanced_tutorials/10_diagnostics_posterior_predictive_checks.html) or [L-C2ST](https://sbi.readthedocs.io/en/latest/how_to_guide/13_diagnostics_lc2st.html). L-C2ST needs posterior samples for many simulated observations, which is fast for NPE. SBC and TARP are also valid, but they need a full sequential run for each test observation.

`kl_divergence_mc` needs a normalized `log_prob()`. It accepts direct, variational and vector field posteriors. For posteriors that sample with MCMC or rejection sampling, compare samples with `sbi.utils.metrics.c2st` instead.

## Example

You can find an example and more explanation in the tutorial [here](https://sbi.readthedocs.io/en/latest/advanced_tutorials/02_multiround_inference.html).
